# Advanced Interface features

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
import random

import gradio as gr


def chat(message, history):
    history = history or []
    if message.startswith("How many"):
        response = random.randint(1, 10)
    elif message.startswith("How"):
        response = random.choice(["Great", "Good", "Okay", "Bad"])
    elif message.startswith("Where"):
        response = random.choice(["Here", "There", "Somewhere"])
    else:
        response = "I don't know"
    history.append((message, response))
    return history, history


# iface = gr.Interface(
#     fn=chat,
#     inputs=[
#         gr.Textbox(label="Message"),
#         gr.State([]),
#     ],
#     outputs=[
#         gr.Chatbot(),
#         gr.State(),
#     ],
#     flagging_mode="never",
# )
iface = gr.ChatInterface(
    fn=chat,
    flagging_mode="never",
)
iface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
# import requests
# import tensorflow as tf

# import gradio as gr

# inception_net = tf.keras.applications.MobileNetV2()  # load the model

# # Download human-readable labels for ImageNet.
# response = requests.get("https://git.io/JJkYN")
# labels = response.text.split("\n")


# def classify_image(inp):
#     inp = inp.reshape((-1, 224, 224, 3))
#     inp = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
#     prediction = inception_net.predict(inp).flatten()
#     return {labels[i]: float(prediction[i]) for i in range(1000)}


# image = gr.Image(shape=(224, 224))
# label = gr.Label(num_top_classes=3)

# title = "Gradio Image Classifiction + Interpretation Example"
# gr.Interface(
#     fn=classify_image, inputs=image, outputs=label, interpretation="default", title=title
# ).launch()

In [3]:
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from PIL import Image
import gradio as gr


# Load pretrained MobileNetV2
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)
model.eval()

# Torchvision provides the correct preprocessing for these weights
preprocess = weights.transforms()

# ImageNet labels
labels = weights.meta["categories"]


def classify_image(inp):
    # Gradio normally gives us a PIL image when type="pil"
    if not isinstance(inp, Image.Image):
        inp = Image.fromarray(inp)

    # Preprocess and add batch dimension:
    # [3, 224, 224] -> [1, 3, 224, 224]
    x = preprocess(inp).unsqueeze(0)

    # Inference
    with torch.no_grad():
        logits = model(x)

    # Convert logits to probabilities
    probabilities = torch.softmax(logits[0], dim=0)

    return {
        labels[i]: float(probabilities[i])
        for i in range(len(labels))
    }


image = gr.Image(type="pil")
label = gr.Label(num_top_classes=3)

title = "Gradio Image Classification + Interpretation Example"

gr.Interface(
    fn=classify_image,
    inputs=image,
    outputs=label,
    title=title,
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
